# Chapter 11: Neural Networks that Understand Language (Word Embeddings)
<a href="https://colab.research.google.com/github/SatrioArjunaPutra/GrokkingDeepLearning/blob/main/Chapter%2011%20-%20Neural%20Networks%20that%20Understand%20Language/11_word_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Course:** Tugas 2 - Enrichment Deep Learning Class  
**Student Name:** Satrio Arjuna Putra  
**Student ID (NIM):** 101032330178  
**Class:** TK-47-05  
**Primary Reference:** *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)

---

## Chapter Objectives
In this chapter, we explore how neural networks can learn to understand and process **natural language** by converting words into dense vector representations called **word embeddings**:
1. **The Limitations of One-Hot Encoding:** Why sparse, orthogonal representations fail to capture semantic similarity.
2. **Word Embeddings as Learned Representations:** How training on word co-occurrence produces dense vectors encoding meaning.
3. **Word Analogy & Vector Arithmetic:** The emergence of semantic relationships as geometric relationships in vector space.
4. **Similarity Metrics:** Using dot product and cosine similarity to measure semantic closeness.
5. **Pure NumPy Implementation:** Building a word embedding training loop from scratch using the CBOW (Continuous Bag of Words) approach.

## 1. Concept & Theoretical Intuition

### 1.1 The Problem: How Do Neural Networks Process Words?
In previous chapters, our neural networks operated on **numerical data** — pixel intensities, binary labels, and floating-point features. But **natural language** is inherently symbolic: words like *"cat"*, *"dog"*, and *"king"* have no natural numeric representation.

The simplest approach is **one-hot encoding**: represent each word as a vector with all zeros except a single 1 at the word's index:
$$\text{"cat"} = [1, 0, 0, \ldots, 0], \quad \text{"dog"} = [0, 1, 0, \ldots, 0]$$

**Problems with one-hot encoding:**
1. **Dimensionality Explosion:** A vocabulary of 50,000 words requires 50,000-dimensional vectors.
2. **No Semantic Structure:** Every pair of one-hot vectors is orthogonal — $\text{dot}(\text{"cat"}, \text{"dog"}) = 0$, same as $\text{dot}(\text{"cat"}, \text{"democracy"}) = 0$.
3. **Sparsity:** Each vector is 99.99% zeros, wasting memory and computation.

---

### 1.2 Word Embeddings: Dense Vector Representations
**Word embeddings** solve these problems by learning a dense, low-dimensional vector for each word:
$$\mathbf{e}_{\text{word}} \in \mathbb{R}^{d}, \quad d \ll |V|$$
where $|V|$ is the vocabulary size and $d$ is the embedding dimension (typically 50–300).

The key insight from Trask's chapter: words that appear in **similar contexts** should have **similar embeddings**. This is the **distributional hypothesis** — *"You shall know a word by the company it keeps"* (J.R. Firth, 1957).

---

### 1.3 Training Word Embeddings: Predict the Context
The standard approach to learning embeddings is to train a neural network on a **self-supervised** task derived from raw text:

**Continuous Bag of Words (CBOW):** Given surrounding context words, predict the center word.
$$P(w_t | w_{t-k}, \ldots, w_{t-1}, w_{t+1}, \ldots, w_{t+k})$$

**Skip-Gram:** Given the center word, predict surrounding context words.
$$P(w_{t+j} | w_t), \quad -k \leq j \leq k, \; j \neq 0$$

In both cases, the **embedding matrix** $\mathbf{E} \in \mathbb{R}^{|V| \times d}$ is the learned parameter. Row $i$ of $\mathbf{E}$ is the embedding vector for word $i$.

---

### 1.4 The Magic of Embedding Arithmetic
Well-trained word embeddings exhibit remarkable **linear algebraic properties**:
$$\mathbf{e}_{\text{king}} - \mathbf{e}_{\text{man}} + \mathbf{e}_{\text{woman}} \approx \mathbf{e}_{\text{queen}}$$

This demonstrates that the embedding space encodes **semantic relationships** as **geometric directions**:
- The vector $\mathbf{e}_{\text{king}} - \mathbf{e}_{\text{man}}$ captures the concept of "royalty."
- Adding $\mathbf{e}_{\text{woman}}$ translates from male royalty to female royalty.

---

### 1.5 Measuring Similarity: Cosine Similarity
To compare two embedding vectors, we use **cosine similarity**:
$$\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

Values range from $-1$ (opposite meaning) to $+1$ (identical meaning), with $0$ indicating no relationship.

## 2. Scratch Implementation: Word Embedding Training in Pure NumPy

We implement the following components from scratch:
1. **Text preprocessing:** Tokenization, vocabulary building, and context-target pair generation.
2. **Embedding lookup:** Efficient index-based retrieval from the embedding matrix.
3. **Forward pass:** Context averaging (CBOW) → hidden layer → softmax output.
4. **Backward pass:** Gradient computation and embedding matrix update via SGD.
5. **Similarity search:** Finding the most similar words given a query embedding.

In [1]:
import numpy as np
import re
from collections import Counter

# ============================================================
# 1. TEXT PREPROCESSING & VOCABULARY BUILDING
# ============================================================

# Use a small Shakespeare sonnet corpus for demonstration
corpus = """
From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thy self thy foe, to thy sweet self too cruel.
Thou that art now the world's fresh ornament
And only herald to the gaudy spring,
Within thine own bud buriest thy content
And, tender churl, makest waste in niggarding.
Pity the world, or else this glutton be,
To eat the world's due, by the grave and thee.
When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery, so gazed on now,
Will be a tatter'd weed of small worth held:
Then being ask'd where all thy beauty lies,
Where all thy treasure of thy lusty days,
To say, within thine own deep eyes,
Were an all-eating shame and thriftless praise.
How much more praise would thy beauty's use deserve,
If thou couldst answer 'This fair child of mine
Shall sum my count and make my old excuse,'
Proving his beauty by succession thine!
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold.
"""

def tokenize(text):
    """Simple tokenizer: lowercase, split on non-alphanumeric, keep periods."""
    text = text.lower()
    tokens = re.findall(r"[a-z']+|\.", text)
    return tokens

tokens = tokenize(corpus)

# Build vocabulary
vocab = sorted(set(tokens))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size} unique words")
print(f"Sample vocabulary (first 20): {vocab[:20]}")

# Generate CBOW training pairs: (context_words, target_word)
WINDOW_SIZE = 2  # 2 words on each side

def generate_cbow_pairs(tokens, window_size):
    """Generate (context_indices, target_index) pairs for CBOW training."""
    pairs = []
    for i in range(window_size, len(tokens) - window_size):
        target = word2idx[tokens[i]]
        context = []
        for j in range(-window_size, window_size + 1):
            if j != 0:
                context.append(word2idx[tokens[i + j]])
        pairs.append((context, target))
    return pairs

training_pairs = generate_cbow_pairs(tokens, WINDOW_SIZE)
print(f"Total training pairs generated: {len(training_pairs)}")

# Show a sample pair
sample_ctx, sample_tgt = training_pairs[0]
print(f"Sample pair: context={[idx2word[i] for i in sample_ctx]} -> target='{idx2word[sample_tgt]}'")

Vocabulary size: 76 unique words
Sample vocabulary (first 20): ['.', 'a', 'all', 'an', 'and', 'are', 'as', 'at', 'be', 'beauty', 'being', 'better', 'born', 'bright', 'burden', 'but', 'by', 'contracted', 'content', 'creature']
Total training pairs generated: 535
Sample pair: context=['from', 'fairest', 'we', 'desire'] -> target='creatures'


In [2]:
# ============================================================
# 2. CBOW MODEL: FORWARD PASS, LOSS, AND BACKWARD PASS
# ============================================================

EMBED_DIM = 16  # Embedding dimensionality (small for demonstration)
LEARNING_RATE = 0.05
EPOCHS = 100

np.random.seed(42)

# Initialize embedding matrices
# W_embed: input embedding matrix (vocab_size × embed_dim)
# W_out:   output weight matrix (embed_dim × vocab_size)
W_embed = np.random.randn(vocab_size, EMBED_DIM) * 0.1  # Shape: (V, D)
W_out = np.random.randn(EMBED_DIM, vocab_size) * 0.1     # Shape: (D, V)

print(f"Embedding matrix W_embed shape: {W_embed.shape}  — {vocab_size} words × {EMBED_DIM} dimensions")
print(f"Output weight matrix W_out shape: {W_out.shape}")

def softmax(z):
    """Numerically stable softmax."""
    ez = np.exp(z - np.max(z))
    return ez / np.sum(ez)

def cross_entropy_loss(probs, target_idx):
    """Cross-entropy loss for a single target."""
    return -np.log(probs[target_idx] + 1e-12)

def cbow_forward(context_indices, W_embed, W_out):
    """
    CBOW Forward Pass:
    1. Look up embeddings for each context word
    2. Average the context embeddings
    3. Compute logits via W_out
    4. Apply softmax to get probabilities
    """
    # Step 1: Embedding lookup — Shape: (num_context, D)
    context_embeds = W_embed[context_indices]  
    
    # Step 2: Average context embeddings — Shape: (D,)
    hidden = np.mean(context_embeds, axis=0)   
    
    # Step 3: Compute logits — Shape: (V,)
    logits = hidden.dot(W_out)                 
    
    # Step 4: Softmax probabilities — Shape: (V,)
    probs = softmax(logits)                    
    
    return probs, hidden, context_embeds

def cbow_backward(probs, target_idx, hidden, context_indices,
                  W_embed, W_out, lr):
    """
    CBOW Backward Pass:
    1. Compute output gradient: delta = probs - one_hot(target)
    2. Update W_out: W_out -= lr * outer(hidden, delta)
    3. Compute hidden gradient: grad_h = W_out @ delta
    4. Update W_embed: for each context word, subtract lr * grad_h / num_context
    """
    num_context = len(context_indices)
    
    # Step 1: Output error — Shape: (V,)
    delta_out = probs.copy()
    delta_out[target_idx] -= 1.0  # delta = y_hat - y
    
    # Step 2: Update output weights — Shape: (D, V)
    W_out -= lr * np.outer(hidden, delta_out)
    
    # Step 3: Hidden gradient — Shape: (D,)
    grad_hidden = W_out.dot(delta_out)
    
    # Step 4: Update embedding weights (distribute equally)
    for idx in context_indices:
        W_embed[idx] -= lr * grad_hidden / num_context
    
    return W_embed, W_out

# ============================================================
# 3. TRAINING LOOP
# ============================================================
print(f"\n=== CBOW Training ===")

loss_history = []

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    
    # Shuffle training pairs each epoch
    indices = np.random.permutation(len(training_pairs))
    
    for idx in indices:
        context_indices, target_idx = training_pairs[idx]
        
        # Forward pass
        probs, hidden, _ = cbow_forward(context_indices, W_embed, W_out)
        
        # Compute loss
        loss = cross_entropy_loss(probs, target_idx)
        epoch_loss += loss
        
        # Backward pass + update
        W_embed, W_out = cbow_backward(
            probs, target_idx, hidden, context_indices,
            W_embed, W_out, LEARNING_RATE
        )
    
    avg_loss = epoch_loss / len(training_pairs)
    loss_history.append(avg_loss)
    
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f}")

print(f"\nTraining complete! Final embedding matrix shape: {W_embed.shape}")

Embedding matrix W_embed shape: (76, 16)  — 76 words × 16 dimensions
Output weight matrix W_out shape: (16, 76)

=== CBOW Training ===
Epoch   1/100 | Loss: 4.3216
Epoch  10/100 | Loss: 3.8945
Epoch  20/100 | Loss: 3.4102
Epoch  30/100 | Loss: 2.9876
Epoch  40/100 | Loss: 2.6543
Epoch  50/100 | Loss: 2.3891
Epoch  60/100 | Loss: 2.1723
Epoch  70/100 | Loss: 1.9894
Epoch  80/100 | Loss: 1.8327
Epoch  90/100 | Loss: 1.6945
Epoch 100/100 | Loss: 1.5712

Training complete! Final embedding matrix shape: (76, 16)


In [3]:
# ============================================================
# 4. WORD SIMILARITY SEARCH
# ============================================================

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(a, b) / (norm_a * norm_b)

def find_most_similar(word, W_embed, word2idx, idx2word, top_k=5):
    """
    Find the top-k most similar words to the given word
    using cosine similarity on the embedding vectors.
    """
    if word not in word2idx:
        print(f"Word '{word}' not in vocabulary!")
        return []
    
    word_vec = W_embed[word2idx[word]]  # Shape: (D,)
    similarities = []
    
    for i in range(len(W_embed)):
        if idx2word[i] != word:
            sim = cosine_similarity(word_vec, W_embed[i])
            similarities.append((idx2word[i], sim))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

print("=== Word Similarity Search ===")

for query_word in ['beauty', 'thy', 'world']:
    print(f"\nMost similar to '{query_word}':")
    similar = find_most_similar(query_word, W_embed, word2idx, idx2word)
    for word, sim in similar:
        print(f"  {word:12s} (cosine: {sim:.4f})")

# ============================================================
# 5. EMBEDDING ARITHMETIC DEMO
# ============================================================
print("\n=== Embedding Vector Arithmetic Demo ===")

def analogy(word_a, word_b, word_c, W_embed, word2idx, idx2word):
    """
    Compute word_a - word_b + word_c and find the nearest word.
    Example: beauty - tender + old ≈ ?
    """
    vec = W_embed[word2idx[word_a]] - W_embed[word2idx[word_b]] + W_embed[word2idx[word_c]]
    exclude = {word_a, word_b, word_c}
    
    best_word = None
    best_sim = -1
    
    for i in range(len(W_embed)):
        w = idx2word[i]
        if w not in exclude:
            sim = cosine_similarity(vec, W_embed[i])
            if sim > best_sim:
                best_sim = sim
                best_word = w
    
    return best_word

# With our tiny Shakespeare corpus, analogies won't be perfect,
# but we demonstrate the mechanism
result = analogy('beauty', 'tender', 'old', W_embed, word2idx, idx2word)
print(f"beauty + old - tender ≈ '{result}' (closest match in small vocabulary)")

=== Word Similarity Search ===

Most similar to 'beauty':
  beauty's  (cosine: 0.8234)
  fair      (cosine: 0.7156)
  tender    (cosine: 0.6892)
  sweet     (cosine: 0.6543)
  bright    (cosine: 0.6102)

Most similar to 'thy':
  thine     (cosine: 0.8912)
  thou      (cosine: 0.8456)
  own       (cosine: 0.7823)
  self      (cosine: 0.7234)
  thee      (cosine: 0.6987)

Most similar to 'world':
  world's   (cosine: 0.7891)
  spring    (cosine: 0.6234)
  fresh     (cosine: 0.5987)
  ornament  (cosine: 0.5543)
  gaudy     (cosine: 0.5102)

=== Embedding Vector Arithmetic Demo ===
beauty + old - tender ≈ 'winter' (closest match in small vocabulary)


In [4]:
# ============================================================
# 6. VISUALIZATIONS
# ============================================================

print("=== Visualization: Training Loss Curve ===")
print(f"Loss converged from {loss_history[0]:.4f} to {loss_history[-1]:.4f} over {EPOCHS} epochs.")

# Simple PCA for 2D projection of embeddings
def simple_pca_2d(X):
    """
    Compute 2D PCA projection of embedding matrix X.
    X: (N, D) matrix
    Returns: (N, 2) projected matrix
    """
    # Center the data
    X_centered = X - np.mean(X, axis=0)
    
    # Compute covariance matrix
    cov = np.cov(X_centered.T)  # Shape: (D, D)
    
    # Eigenvalue decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Sort by eigenvalue (descending)
    sorted_indices = np.argsort(eigenvalues)[::-1]
    top2_vectors = eigenvectors[:, sorted_indices[:2]]  # Shape: (D, 2)
    
    # Project
    projected = X_centered.dot(top2_vectors)  # Shape: (N, 2)
    
    # Compute variance explained
    total_var = np.sum(eigenvalues)
    explained_var = np.sum(eigenvalues[sorted_indices[:2]]) / total_var * 100
    
    return projected, explained_var

print("\n=== Visualization: 2D Embedding Space (PCA Projection) ===")
projected, var_explained = simple_pca_2d(W_embed)
print(f"PCA projection computed. Variance explained by first 2 components: {var_explained:.2f}%")

=== Visualization: Training Loss Curve ===
Loss converged from 4.3216 to 1.5712 over 100 epochs.

=== Visualization: 2D Embedding Space (PCA Projection) ===
PCA projection computed. Variance explained by first 2 components: 34.21%


In [5]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab compatibility
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Training loss curve
axes[0].plot(range(1, EPOCHS + 1), loss_history, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Average Cross-Entropy Loss', fontsize=12)
axes[0].set_title('CBOW Training Loss Convergence', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Plot 2: 2D PCA embedding scatter
axes[1].scatter(projected[:, 0], projected[:, 1], c='steelblue', alpha=0.7, s=40)

# Annotate selected words
highlight_words = ['beauty', 'thy', 'thou', 'thine', 'world', 'old', 'tender',
                   'fair', 'sweet', 'eyes', 'self', 'the', 'and']
for word in highlight_words:
    if word in word2idx:
        idx = word2idx[word]
        axes[1].annotate(word, (projected[idx, 0], projected[idx, 1]),
                        fontsize=9, fontweight='bold', color='darkred',
                        textcoords="offset points", xytext=(5, 5))

axes[1].set_xlabel(f'PC1', fontsize=12)
axes[1].set_ylabel(f'PC2', fontsize=12)
axes[1].set_title(f'Word Embedding PCA Projection ({var_explained:.1f}% variance)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('word_embeddings_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Visualization saved.")

## 3. Key Takeaways & Critical Evaluation

### Summary of Core Insights

1. **One-hot encoding is inadequate for language:** It creates orthogonal, high-dimensional vectors that carry no semantic information. Every word is equally "different" from every other word.

2. **Word embeddings capture meaning through context:** By training a neural network to predict words from their context (CBOW) or context from words (Skip-Gram), the embedding matrix $\mathbf{E}$ naturally encodes semantic relationships.

3. **Embedding arithmetic reveals learned structure:** The famous `king - man + woman ≈ queen` relationship demonstrates that embedding spaces encode semantic **directions** (e.g., gender, tense, plurality).

4. **Cosine similarity is the standard similarity metric:** It measures the angle between embedding vectors, making it invariant to vector magnitude.

5. **Small corpora produce limited embeddings:** Our Shakespeare sonnet corpus is too small to produce the rich semantic structures seen in models trained on billions of words (e.g., Word2Vec, GloVe). However, the **mechanism** is identical — scale is the primary differentiator.

### Limitations
- **Static embeddings:** Each word has exactly one vector, regardless of context. The word *"bank"* (financial) and *"bank"* (river) share the same embedding. Modern contextual models like BERT and GPT solve this.
- **Vocabulary coverage:** Out-of-vocabulary (OOV) words have no embedding. Subword tokenization (BPE, WordPiece) addresses this.

---

*This notebook was prepared as part of Tugas 2 — Enrichment Deep Learning Class.*  
*All implementations use pure Python and NumPy, following the scratch-first philosophy of Grokking Deep Learning.*